# Module 1 — Data Exploration: MovieLens 25M
**Smart Recommendation System · Data foundation, exploration & preprocessing**

This notebook demonstrates the Module 1 pipeline on the raw MovieLens 25M dataset:
loading, schema inspection, missing values, duplicates, rating/user/movie/genre/tag
analysis, and the final data-quality summary.

Raw files in `data/raw/ml-25m/` are read-only; all outputs live under `data/processed/`
and `outputs/`.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
for candidate in (PROJECT_ROOT, Path.cwd().resolve()):
    if (candidate / "src").is_dir() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

from src import config
from src.data_loader import (load_movies, load_ratings, load_tags, load_links,
                             load_genome_scores, load_genome_tags)
from src import analysis, data_quality as dq
from src.data_preprocessor import DataPreprocessor, extract_release_year

import pandas as pd

movies = load_movies()
ratings = load_ratings()
tags = load_tags()
links = load_links()
genome_scores = load_genome_scores()
genome_tags = load_genome_tags()

DATASETS = {"movies": movies, "ratings": ratings, "tags": tags,
            "links": links, "genome-scores": genome_scores, "genome-tags": genome_tags}
print({name: df.shape for name, df in DATASETS.items()})

2026-08-22 12:33:29 | INFO     | src.data_loader | Loading 'movies' from D:\New Project\CodeVedex Projects\CODEVEDX\Project-05-Smart-Recommendation-System\data\raw\ml-25m\movies.csv ...


2026-08-22 12:33:29 | INFO     | src.data_loader | Loaded 'movies': 62423 rows x 3 columns (8.4 MiB in memory)


2026-08-22 12:33:29 | INFO     | src.data_loader | Loading 'ratings' from D:\New Project\CodeVedex Projects\CODEVEDX\Project-05-Smart-Recommendation-System\data\raw\ml-25m\ratings.csv ...


2026-08-22 12:33:58 | INFO     | src.data_loader | Loaded 'ratings': 25000095 rows x 4 columns (476.8 MiB in memory)


2026-08-22 12:33:58 | INFO     | src.data_loader | Loading 'tags' from D:\New Project\CodeVedex Projects\CODEVEDX\Project-05-Smart-Recommendation-System\data\raw\ml-25m\tags.csv ...


...

2026-08-22 12:34:05 | INFO     | src.data_loader | Loading 'genome_scores' from D:\New Project\CodeVedex Projects\CODEVEDX\Project-05-Smart-Recommendation-System\data\raw\ml-25m\genome-scores.csv ...


2026-08-22 12:34:21 | INFO     | src.data_loader | Loaded 'genome_scores': 15584448 rows x 3 columns (148.6 MiB in memory)


2026-08-22 12:34:21 | INFO     | src.data_loader | Loading 'genome_tags' from D:\New Project\CodeVedex Projects\CODEVEDX\Project-05-Smart-Recommendation-System\data\raw\ml-25m\genome-tags.csv ...


2026-08-22 12:34:21 | INFO     | src.data_loader | Loaded 'genome_tags': 1128 rows x 2 columns (0.1 MiB in memory)


{'movies': (62423, 3), 'ratings': (25000095, 4), 'tags': (1093360, 4), 'links': (62423, 3), 'genome-scores': (15584448, 3), 'genome-tags': (1128, 2)}


## 1–2. Dataset loading & schema inspection

In [2]:
schema_summary = pd.DataFrame(
    {name: dq.validate_schema(df, name.replace("-", "_"))["valid"] for name, df in {
        "movies": movies, "ratings": ratings, "tags": tags,
        "links": links, "genome_scores": genome_scores, "genome_tags": genome_tags,
    }.items()},
    index=["schema_valid"],
).T
display(schema_summary)

profiles = pd.DataFrame(
    {name: dq.profile_dataset(df, name) for name, df in {
        "movies": movies, "ratings": ratings, "tags": tags,
        "links": links, "genome-scores": genome_scores, "genome-tags": genome_tags,
    }.items()}
).T[["rows", "columns", "memory_mb", "duplicate_rows"]]
profiles

2026-08-22 12:34:21 | INFO     | src.data_quality | Schema check passed for 'movies'


2026-08-22 12:34:21 | INFO     | src.data_quality | Schema check passed for 'ratings'


2026-08-22 12:34:21 | INFO     | src.data_quality | Schema check passed for 'tags'


2026-08-22 12:34:21 | INFO     | src.data_quality | Schema check passed for 'links'


2026-08-22 12:34:21 | INFO     | src.data_quality | Schema check passed for 'genome_scores'


...

2026-08-22 12:35:33 | INFO     | src.data_quality | Profiled 'tags': 1093360 rows, 4 cols, 79.98 MiB, 1 columns with missing values, 0 duplicate rows


2026-08-22 12:35:33 | INFO     | src.data_quality | Profiled 'links': 62423 rows, 3 cols, 1.31 MiB, 1 columns with missing values, 0 duplicate rows


2026-08-22 12:35:50 | INFO     | src.data_quality | Profiled 'genome-scores': 15584448 rows, 3 cols, 148.63 MiB, 0 columns with missing values, 0 duplicate rows


2026-08-22 12:35:50 | INFO     | src.data_quality | Profiled 'genome-tags': 1128 rows, 2 cols, 0.07 MiB, 0 columns with missing values, 0 duplicate rows


,rows,columns,memory_mb,duplicate_rows
movies,62423,3,8.43,0
ratings,25000095,4,476.84,0
tags,1093360,4,79.98,0
links,62423,3,1.31,0
genome-scores,15584448,3,148.63,0
genome-tags,1128,2,0.07,0


## 3. Missing-value analysis

In [3]:
missing_rows = []
for name, df in DATASETS.items():
    for col, info in dq.analyze_missing_values(df, name)["by_column"].items():
        if info["missing"] > 0:
            missing_rows.append({"dataset": name, "column": col,
                                 "missing": info["missing"], "pct": info["pct"]})
missing_df = pd.DataFrame(missing_rows).set_index(["dataset", "column"])
print(f"Total missing cells across all raw datasets: "
      f"{sum(int(df.isna().sum().sum()) for df in DATASETS.values()):,}")
missing_df

2026-08-22 12:35:50 | INFO     | src.data_quality | Missing-value scan for 'movies': 0 missing cells overall


2026-08-22 12:35:51 | INFO     | src.data_quality | Missing-value scan for 'ratings': 0 missing cells overall


2026-08-22 12:35:51 | INFO     | src.data_quality | Missing-value scan for 'tags': 16 missing cells overall


2026-08-22 12:35:51 | INFO     | src.data_quality | Missing-value scan for 'links': 107 missing cells overall


2026-08-22 12:35:52 | INFO     | src.data_quality | Missing-value scan for 'genome-scores': 0 missing cells overall


2026-08-22 12:35:52 | INFO     | src.data_quality | Missing-value scan for 'genome-tags': 0 missing cells overall


Total missing cells across all raw datasets: 123


,,missing,pct
dataset,column,,
tags,tag,16,0.0015
links,tmdbId,107,0.1714


Missing-value handling rules (applied during cleaning):
- `tags.tag` null/empty → record dropped (a tag without text has no value)
- `tags.timestamp` missing → **retained** as `<NA>` (tag text still valuable)
- `links.imdbId/tmdbId` missing → legitimate, retained as `<NA>`
- `ratings` required fields / `movies.title` → invalid records dropped

## 4. Duplicate analysis

In [4]:
dup_movies = dq.analyze_movie_duplicates(movies)
dup_ratings = dq.analyze_rating_duplicates(ratings)
dup_tags = dq.analyze_tag_duplicates(tags)

pd.DataFrame({
    "check": ["movies duplicate movieId", "movies duplicate title (distinct ids)",
              "ratings exact repeats (user+movie+timestamp)",
              "ratings repeated user+movie re-ratings",
              "tags exact repeats",
              "links duplicate movieId"],
    "count": [dup_movies["duplicate_movieId"]["duplicate_records"],
              dup_movies["duplicate_title"]["duplicate_records"],
              dup_ratings["exact_duplicates"]["duplicate_records"],
              dup_ratings["repeated_user_movie_interactions"]["duplicate_records"],
              dup_tags["exact_duplicates"]["duplicate_records"],
              dq.analyze_link_duplicates(links)["duplicate_movieId"]["duplicate_records"]],
}).set_index("check")

,count
check,
movies duplicate movieId,0
movies duplicate title (distinct ids),98
ratings exact repeats (user+movie+timestamp),0
ratings repeated user+movie re-ratings,0
tags exact repeats,0
links duplicate movieId,0


In [5]:
# Duplicate titles with distinct movieIds are legitimate re-issues - retained.
dup_title_counts = movies["title"].value_counts()
movies[movies["title"].isin(dup_title_counts[dup_title_counts > 1].index)].head(6)

,movieId,title,genres
580,588,Aladdin (1992),Adventure|Animation|Children|Comedy|Musical
1710,1788,Men with Guns (1997),Action|Drama
2553,2644,Dracula (1931),Horror
2759,2851,Saturn 3 (1980),Adventure|Sci-Fi|Thriller
3454,3553,Gossip (2000),Drama|Thriller
3499,3598,Hamlet (2000),Crime|Drama|Romance|Thriller


## 5. Rating analysis

In [6]:
pd.concat([
    pd.Series(analysis.analyze_ratings(ratings), name="value"),
    pd.Series(analysis.rating_distribution(ratings)["counts_by_value"], name="rating_value"),
]).to_frame().T

2026-08-22 12:37:03 | INFO     | src.analysis | Rating stats: n=25000095, mean=3.534, median=3.500, std=1.061


2026-08-22 12:37:04 | INFO     | src.analysis | Rating distribution computed across 10 grid values


,total_ratings,min,max,mean,median,std,0.5,1.0,1.5,2.0,2.5,3.0,3.5,4.0,4.5,5.0
0,25000095.0,0.5,5.0,3.5339,3.5,1.0607,393068.0,776815.0,399490.0,1640868.0,1262797.0,4896928.0,3177318.0,6639798.0,2200539.0,3612474.0


In [7]:
from src.visualization import plot_rating_distribution
plot_rating_distribution(ratings);

2026-08-22 12:37:09 | INFO     | src.visualization | Saved chart D:\New Project\CodeVedex Projects\CODEVEDX\Project-05-Smart-Recommendation-System\outputs\charts\rating_distribution.png


## 6. User activity analysis

In [8]:
user_stats = analysis.analyze_users(ratings)
pd.Series(user_stats, name="user statistics").to_frame()

2026-08-22 12:37:11 | INFO     | src.analysis | User activity: 162541 users, mean 153.8, median 71 ratings/user, 1627 highly active (>= 1228), 4611 low activity (<= 20)


,user statistics
unique_users,162541.0000
ratings_per_user_min,20.0000
ratings_per_user_max,32202.0000
ratings_per_user_mean,153.8079
ratings_per_user_median,71.0000
highly_active_users_ge_99th_pct,1627.0000
high_activity_threshold_99th_pct,1228.0000
low_activity_users_le_1st_pct,4611.0000
low_activity_threshold_1st_pct,20.0000


In [9]:
from src.visualization import plot_user_activity_distribution
plot_user_activity_distribution(ratings);

2026-08-22 12:37:16 | INFO     | src.visualization | Saved chart D:\New Project\CodeVedex Projects\CODEVEDX\Project-05-Smart-Recommendation-System\outputs\charts\user_activity_distribution.png


## 7. Movie popularity analysis

In [10]:
movie_stats = analysis.analyze_movie_popularity(ratings, movies)
{k: v for k, v in movie_stats.items() if k != "top_movies_by_rating_count"}

2026-08-22 12:37:20 | INFO     | src.analysis | Movie popularity: 59047 movies rated, mean 423.4 ratings/movie, max 81491


{'unique_movies_rated': 59047,
 'ratings_per_movie_min': 1,
 'ratings_per_movie_max': 81491,
 'ratings_per_movie_mean': 423.3931,
 'ratings_per_movie_median': 6.0}

In [11]:
top10 = pd.DataFrame(movie_stats["top_movies_by_rating_count"]).head(10)
top10.set_index("rank")[["title", "rating_count", "mean_rating"]]

,title,rating_count,mean_rating
rank,,,
1,Forrest Gump (1994),81491,4.0480
2,"Shawshank Redemption, The (1994)",81482,4.4136
3,Pulp Fiction (1994),79672,4.1889
4,"Silence of the Lambs, The (1991)",74127,4.1513
5,"Matrix, The (1999)",72674,4.1541
6,Star Wars: Episode IV - A New Hope (1977),68717,4.1202
7,Jurassic Park (1993),64144,3.6792
8,Schindler's List (1993),60411,4.2476
9,Braveheart (1995),59184,4.0023


In [12]:
from src.visualization import plot_movie_popularity_distribution
plot_movie_popularity_distribution(ratings);

2026-08-22 12:37:25 | INFO     | src.visualization | Saved chart D:\New Project\CodeVedex Projects\CODEVEDX\Project-05-Smart-Recommendation-System\outputs\charts\movie_popularity_distribution.png


## 8. Genre analysis

In [13]:
genre_stats = analysis.analyze_genres(movies)
genre_table = pd.DataFrame(genre_stats["movies_by_genre"]).T
genre_table.index.name = "genre"
print(f"distinct genres: {genre_stats['num_distinct_genres']} | "
      f"no-genre movies: {genre_stats['movies_with_no_genre']:,} | "
      f"avg genres/movie: {genre_stats['avg_genres_per_movie']}")
genre_table

2026-08-22 12:37:25 | INFO     | src.analysis | Genres: 19 distinct genres, 5062 movies without genres (8.11%), avg 1.72 genres/movie


distinct genres: 19 | no-genre movies: 5,062 | avg genres/movie: 1.718


,count,pct_of_movies
genre,,
Drama,25606.0,41.020
Comedy,16870.0,27.025
Thriller,8654.0,13.863
Romance,7719.0,12.366
Action,7348.0,11.771
Horror,5989.0,9.594
Documentary,5605.0,8.979
Crime,5319.0,8.521
(no genres listed),5062.0,8.109


In [14]:
from src.visualization import plot_genre_distribution
plot_genre_distribution(genre_stats);

2026-08-22 12:37:27 | INFO     | src.visualization | Saved chart D:\New Project\CodeVedex Projects\CODEVEDX\Project-05-Smart-Recommendation-System\outputs\charts\genre_distribution.png


## 9. Tag analysis

In [15]:
tag_stats = analysis.analyze_tags(tags)
{ k: v for k, v in tag_stats.items() if k != "most_frequent_tags" }

2026-08-22 12:37:30 | INFO     | src.analysis | Tags: 1093360 records, 65403 unique tags (case-insensitive)


{'total_tag_records': 1093360,
 'unique_users_tagging': 14592,
 'unique_movies_tagged': 45251,
 'unique_tags_case_insensitive': 65403}

In [16]:
from src.visualization import plot_top_tags
plot_top_tags(analysis.tag_frequency_series(tags));

2026-08-22 12:37:35 | INFO     | src.visualization | Saved chart D:\New Project\CodeVedex Projects\CODEVEDX\Project-05-Smart-Recommendation-System\outputs\charts\top_tags.png


## 10. Cleaning & final data-quality summary

Cleaning applies documented rules only; removal counts are reported below.
Genome datasets are validated here but not duplicated into `data/processed`
(~435 MiB) until a later module needs them.

In [17]:
pre = DataPreprocessor()
movies_clean = pre.clean_movies(movies)
catalog_ids = pd.Index(movies_clean["movieId"].unique())
ratings_clean = pre.clean_ratings(ratings, valid_movie_ids=catalog_ids)
tags_clean = pre.clean_tags(tags)
links_clean = pre.clean_links(links, valid_movie_ids=catalog_ids)
features_base = pre.build_movies_features_base(movies_clean)
print(f"removed {pre.total_removed} rows total; "
      f"features_base shape: {features_base.shape}")

2026-08-22 12:37:36 | INFO     | src.data_preprocessor | [movies] drop missing movieId or blank title -> removed 0, retained 62423


2026-08-22 12:37:36 | INFO     | src.data_preprocessor | [movies] fill missing genres with '(no genres listed)' -> removed 0, retained 62423


2026-08-22 12:37:36 | INFO     | src.data_preprocessor | [movies] duplicate movieId kept-first removal -> removed 0, retained 62423


2026-08-22 12:38:31 | INFO     | src.data_preprocessor | [ratings] drop rows missing required fields -> removed 0, retained 25000095


2026-08-22 12:38:31 | INFO     | src.data_preprocessor | [ratings] drop out-of-range / off-grid ratings -> removed 0, retained 25000095


...

2026-08-22 12:38:36 | INFO     | src.data_preprocessor | [links] retain missing external IDs (<NA>) -> removed 0, retained 62423


2026-08-22 12:38:36 | INFO     | src.data_preprocessor | [links] orphan movieId removal vs. catalog -> removed 0, retained 62423


2026-08-22 12:38:36 | INFO     | src.data_preprocessor | [links] 107 rows have at least one missing external ID


2026-08-22 12:38:38 | INFO     | src.data_preprocessor | [movies_features_base] deterministic normalization (no rows changed) -> removed 0, retained 62423


removed 25 rows total; features_base shape: (62423, 25)


In [18]:
cleaning_df = pd.DataFrame(pre.cleaning_log)
cleaning_df[cleaning_df["records_removed"] > 0].set_index(["dataset", "rule"])

records_removed  \
dataset rule                                                    
tags    drop rows missing userId/movieId/tag               16   
        drop empty-after-trim tags                          9   

                                              records_retained  
dataset rule                                                    
tags    drop rows missing userId/movieId/tag           1093335  
        drop empty-after-trim tags                     1093335

The authoritative quality gate lives in the generated reports:
`outputs/reports/dataset_quality_report.txt` and `.json`. The pipeline
(`python -m src.pipeline`) recomputes them and refuses to finish unless every
critical check passes. Final status of the last run:

In [19]:
import json
report = json.loads(config.QUALITY_REPORT_JSON_PATH.read_text(encoding="utf-8"))
print(f"Final status: {report['status']}")
pd.DataFrame({
    name: {"rows": report[name]["rows"], "columns": report[name]["columns"]}
    for name in ("movies", "ratings", "tags", "links", "genome_scores", "genome_tags")
})

Final status: PASS


,movies,ratings,tags,links,genome_scores,genome_tags
rows,62423,25000095,1093360,62423,15584448,1128
columns,3,4,4,3,3,2
